In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
#for dirname, _, filenames in os.walk('/kaggle/input'):
#    for filename in filenames:
#        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

### Setup paths + create new dataset folder

In [1]:
import os

# RSNA Kaggle input root
rsna_root = "/kaggle/input/rsna-intracranial-aneurysm-detection"

# nnU-Net dirs (we will fill these)
nnunet_raw = "/kaggle/working/nnUNet_raw"
dataset_id = 201
dataset_name = "AneurysmCTA"
dataset_folder = f"Dataset{dataset_id:03d}_{dataset_name}"

dataset_dir = os.path.join(nnunet_raw, dataset_folder)
imagesTr = os.path.join(dataset_dir, "imagesTr")
labelsTr = os.path.join(dataset_dir, "labelsTr")

os.makedirs(imagesTr, exist_ok=True)
os.makedirs(labelsTr, exist_ok=True)

print("imagesTr:", imagesTr)
print("labelsTr:", labelsTr)

imagesTr: /kaggle/working/nnUNet_raw/Dataset201_AneurysmCTA/imagesTr
labelsTr: /kaggle/working/nnUNet_raw/Dataset201_AneurysmCTA/labelsTr


### Load CSVs and select CTA series

In [2]:
import pandas as pd

train_df = pd.read_csv(os.path.join(rsna_root, "train.csv"))
loc_df   = pd.read_csv(os.path.join(rsna_root, "train_localizers.csv"))

# CTA only
cta_df = train_df[train_df["Modality"] == "CTA"].copy()
cta_df["series_id"] = cta_df["SeriesInstanceUID"].astype(str)

# Keep only CTA series that actually have a DICOM folder
series_root = os.path.join(rsna_root, "series")
cta_series_uids = []
for uid in cta_df["series_id"].unique():
    if os.path.isdir(os.path.join(series_root, uid)):
        cta_series_uids.append(uid)

cta_series_uids = sorted(cta_series_uids)
print("Total CTA series with DICOM folder:", len(cta_series_uids))

# Filter localizers to those CTA series
loc_df["SeriesInstanceUID"] = loc_df["SeriesInstanceUID"].astype(str)
loc_df["SOPInstanceUID"]    = loc_df["SOPInstanceUID"].astype(str)

loc_cta = loc_df[loc_df["SeriesInstanceUID"].isin(cta_series_uids)].copy()
print("Localizer rows for CTA:", len(loc_cta))

# POSITIVE series (have at least one localizer)
pos_series_uids = sorted(loc_cta["SeriesInstanceUID"].unique())
pos_set = set(pos_series_uids)
print("CTA series with aneurysm localizers (positives):", len(pos_series_uids))

# NEGATIVE series = CTA series with no localizers
neg_candidates = sorted(set(cta_series_uids) - pos_set)
print("CTA series with no localizers (negatives):", len(neg_candidates))

Total CTA series with DICOM folder: 1808
Localizer rows for CTA: 1220
CTA series with aneurysm localizers (positives): 973
CTA series with no localizers (negatives): 835


### choose 100 dataset first

In [4]:
import numpy as np

rng = np.random.default_rng(42)  # reproducible

# !adjust later, how many of each you want
n_pos_keep = min(50, len(pos_series_uids))
n_neg_keep = min(50, len(neg_candidates))

# sample positives
selected_pos = list(rng.choice(pos_series_uids, size=n_pos_keep, replace=False))

# sample negatives
selected_neg = list(rng.choice(neg_candidates, size=n_neg_keep, replace=False))

selected_series = sorted(set(selected_pos) | set(selected_neg))

print("Selected total series:", len(selected_series))
print("  positives:", len(selected_pos))
print("  negatives:", len(selected_neg))

Selected total series: 100
  positives: 50
  negatives: 50


### Convert each CTA DICOM series → NIfTI in imagesTr

In [5]:
!pip install dicom2nifti

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 110.1 MB/s eta 0:00:0000:0100:01


In [6]:
import dicom2nifti
import glob
import nibabel as nib
import numpy as np

def convert_series_to_nifti(series_uid):
    series_dir = os.path.join(series_root, series_uid)
    out_tmp = "/kaggle/working/tmp_nifti"
    os.makedirs(out_tmp, exist_ok=True)

    # clear tmp
    for f in glob.glob(os.path.join(out_tmp, "*")):
        os.remove(f)

    try:
        dicom2nifti.convert_directory(
            series_dir,
            out_tmp,
            compression=True,
            reorient=True
        )
    except Exception as e:
        print(f"[SKIP IMG] {series_uid}: conversion error -> {e}")
        return False

    nifti_candidates = glob.glob(os.path.join(out_tmp, "*.nii*"))
    if not nifti_candidates:
        print(f"[SKIP IMG] {series_uid}: no NIfTI produced")
        return False

    src_img_path = max(nifti_candidates, key=os.path.getsize)
    target_img = os.path.join(imagesTr, f"{series_uid}_0000.nii.gz")
    os.replace(src_img_path, target_img)

    print(f"[IMG] {series_uid} -> {target_img}")
    return True

from tqdm import tqdm

converted = []
for uid in tqdm(selected_series):
    if convert_series_to_nifti(uid):
        converted.append(uid)

print("Successfully converted images:", len(converted))

  1%|          | 1/100 [00:19<32:57, 19.97s/it]

[IMG] 1.2.826.0.1.3680043.8.498.10163482612339017493097015030860956863 -> /kaggle/working/nnUNet_raw/Dataset201_AneurysmCTA/imagesTr/1.2.826.0.1.3680043.8.498.10163482612339017493097015030860956863_0000.nii.gz


  2%|▏         | 2/100 [00:37<30:07, 18.44s/it]

[IMG] 1.2.826.0.1.3680043.8.498.10454754803302367695534484904787098586 -> /kaggle/working/nnUNet_raw/Dataset201_AneurysmCTA/imagesTr/1.2.826.0.1.3680043.8.498.10454754803302367695534484904787098586_0000.nii.gz


  3%|▎         | 3/100 [01:16<45:04, 27.88s/it]

...

[IMG] 1.2.826.0.1.3680043.8.498.97727219579665267472896437169007074146 -> /kaggle/working/nnUNet_raw/Dataset201_AneurysmCTA/imagesTr/1.2.826.0.1.3680043.8.498.97727219579665267472896437169007074146_0000.nii.gz


 99%|█████████▉| 99/100 [35:05<00:23, 23.50s/it]

[IMG] 1.2.826.0.1.3680043.8.498.98355778619866516223253151047695823055 -> /kaggle/working/nnUNet_raw/Dataset201_AneurysmCTA/imagesTr/1.2.826.0.1.3680043.8.498.98355778619866516223253151047695823055_0000.nii.gz


100%|██████████| 100/100 [35:37<00:00, 21.37s/it]

[IMG] 1.2.826.0.1.3680043.8.498.98740970587848121926125661028314726116 -> /kaggle/working/nnUNet_raw/Dataset201_AneurysmCTA/imagesTr/1.2.826.0.1.3680043.8.498.98740970587848121926125661028314726116_0000.nii.gz
Successfully converted images: 100


## Build corresponding labelsTr for each converted series

In [11]:
# 1) Build SOP→slice index mapping + sphere mask for positives (with localizers)

import pydicom
import ast

# Limit loc_cta to converted series
loc_cta = loc_cta[loc_cta["SeriesInstanceUID"].isin(converted)].copy()
series_with_ann = sorted(loc_cta["SeriesInstanceUID"].unique())
print("Converted CTA series with annotations AND images:", len(series_with_ann))

def build_label_for_series(series_uid, radius=5):
    img_path = os.path.join(imagesTr, f"{series_uid}_0000.nii.gz")
    if not os.path.exists(img_path):
        print(f"[WARN] no image for {series_uid}")
        return False

    img_nii = nib.load(img_path)
    img = img_nii.get_fdata()
    Z, Y, X = img.shape

    # DICOM slice mapping
    series_dir = os.path.join(series_root, series_uid)
    dcm_paths = sorted(glob.glob(os.path.join(series_dir, "*.dcm")))
    if not dcm_paths:
        print(f"[WARN] no DICOMs for {series_uid}")
        return False

    sop_to_idx = {}
    for idx, p in enumerate(dcm_paths):
        ds = pydicom.dcmread(p, stop_before_pixels=True)
        sop_to_idx[ds.SOPInstanceUID] = idx

    label = np.zeros((Z, Y, X), dtype=np.uint8)

    rows = loc_cta[loc_cta["SeriesInstanceUID"] == series_uid]
    for _, row in rows.iterrows():
        sop_uid = row["SOPInstanceUID"]
        if sop_uid not in sop_to_idx:
            print(f"  [WARN] SOP {sop_uid} missing in series {series_uid}")
            continue

        cz = sop_to_idx[sop_uid]

        coords = ast.literal_eval(row["coordinates"])
        x_val = float(coords["x"])
        y_val = float(coords["y"])

        cx = int(round(x_val))
        cy = int(round(y_val))
        cx = max(0, min(X - 1, cx))
        cy = max(0, min(Y - 1, cy))

        # sphere
        z, y, x = np.ogrid[0:Z, 0:Y, 0:X]
        mask = (z - cz)**2 + (y - cy)**2 + (x - cx)**2 <= radius**2
        label[mask] = 1

    out_path = os.path.join(labelsTr, f"{series_uid}.nii.gz")
    lab_nii = nib.Nifti1Image(label, img_nii.affine)
    nib.save(lab_nii, out_path)

    print(f"[POS] {series_uid}: positives={int(label.sum())}")
    return True

for uid in tqdm(series_with_ann):
    build_label_for_series(uid)

Converted CTA series with annotations AND images: 50


  2%|▏         | 1/50 [00:04<03:25,  4.20s/it]

[POS] 1.2.826.0.1.3680043.8.498.10743199796364362163988736837321335182: positives=1111


  4%|▍         | 2/50 [00:13<05:47,  7.23s/it]

[POS] 1.2.826.0.1.3680043.8.498.10777851323461603684026638438811329191: positives=0


...

[POS] 1.2.826.0.1.3680043.8.498.87716964627763925393689510568869137910: positives=298


 98%|█████████▊| 49/50 [04:48<00:08,  8.05s/it]

[POS] 1.2.826.0.1.3680043.8.498.91668463495915545476120070184577733779: positives=515


100%|██████████| 50/50 [04:55<00:00,  5.91s/it]

[POS] 1.2.826.0.1.3680043.8.498.96075577397570658749511147623383862317: positives=515


In [8]:
# check one label

test_label_path = os.path.join(labelsTr, series_with_ann[0] + ".nii.gz")
lab = nib.load(test_label_path).get_fdata()
print("Unique values:", np.unique(lab))   # should be [0. 1.]

Unique values: [0. 1.]


In [9]:
# 2) Build all-zero labels for negatives

series_with_ann_set = set(series_with_ann)
converted_set = set(converted)

neg_uids = sorted(converted_set - series_with_ann_set)
print("Converted series with no annotations (negatives):", len(neg_uids))

for uid in neg_uids:
    img_path = os.path.join(imagesTr, f"{uid}_0000.nii.gz")
    img_nii = nib.load(img_path)
    img = img_nii.get_fdata()
    Z, Y, X = img.shape

    label = np.zeros((Z, Y, X), dtype=np.uint8)
    out_path = os.path.join(labelsTr, f"{uid}.nii.gz")
    lab_nii = nib.Nifti1Image(label, img_nii.affine)
    nib.save(lab_nii, out_path)

    print(f"[NEG] {uid}: saved all-zero label")

Converted series with no annotations (negatives): 50
[NEG] 1.2.826.0.1.3680043.8.498.10163482612339017493097015030860956863: saved all-zero label
[NEG] 1.2.826.0.1.3680043.8.498.10454754803302367695534484904787098586: saved all-zero label
[NEG] 1.2.826.0.1.3680043.8.498.10488876862972997983660376855639751518: saved all-zero label
[NEG] 1.2.826.0.1.3680043.8.498.10489902145908525186969095759982595916: saved all-zero label
[NEG] 1.2.826.0.1.3680043.8.498.11161204043710023971639881771532046119: saved all-zero label
[NEG] 1.2.826.0.1.3680043.8.498.11304226817806458732827210015610897142: saved all-zero label
[NEG] 1.2.826.0.1.3680043.8.498.11653443639338516828198177527745282091: saved all-zero label
[NEG] 1.2.826.0.1.3680043.8.498.11882868066454305806648918521898101299: saved all-zero label
[NEG] 1.2.826.0.1.3680043.8.498.11925824706663452170630610615836381172: saved all-zero label
[NEG] 1.2.826.0.1.3680043.8.498.12594767856833866929395619688146373539: saved all-zero label
[NEG] 1.2.826.0.1

In [12]:
# final check

imagesTr_files = [f for f in os.listdir(imagesTr) if f.endswith("_0000.nii.gz")]
labelsTr_files = [f for f in os.listdir(labelsTr) if f.endswith(".nii.gz")]

print("imagesTr count:", len(imagesTr_files))
print("labelsTr count:", len(labelsTr_files))

imagesTr count: 100
labelsTr count: 100


## Generate dataset.json for nnU-Net

In [14]:
!pip install nnunetv2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 kB 4.8 MB/s eta 0:00:00a 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 5.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 113.1 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 85.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 46.2 MB/s eta 0:00:00
   ━━━

In [15]:
from nnunetv2.dataset_conversion.generate_dataset_json import generate_dataset_json
import os

dataset_dir = "/kaggle/working/nnUNet_raw/Dataset201_AneurysmCTA"
labelsTr_dir = os.path.join(dataset_dir, "labelsTr")

num_training = sum(f.endswith(".nii.gz") for f in os.listdir(labelsTr_dir))
print("num_training (from labelsTr):", num_training)

generate_dataset_json(
    output_folder=dataset_dir,
    channel_names={0: "CTA"},   # single input channel
    labels={
        "background": 0,
        "aneurysm": 1
    },
    num_training_cases=num_training,
    file_ending=".nii.gz",
    dataset_name="AneurysmCTA_100cases",
    description="RSNA CTA aneurysm dataset (~100 cases, sphere masks from train_localizers)"
)

num_training (from labelsTr): 100


In [16]:
import json

with open(os.path.join(dataset_dir, "dataset.json")) as f:
    ds = json.load(f)

print("labels:", ds["labels"])
print("numTraining:", ds["numTraining"])

labels: {'background': 0, 'aneurysm': 1}
numTraining: 100


### 100 cases are too larger for 19.5GB output, shrink dataset sizes

In [21]:
# Classify current 100 cases into positives / negatives
import os
import nibabel as nib
import numpy as np

dataset_dir = "/kaggle/working/nnUNet_raw/Dataset201_AneurysmCTA"
imagesTr = os.path.join(dataset_dir, "imagesTr")
labelsTr = os.path.join(dataset_dir, "labelsTr")

label_files = [f for f in os.listdir(labelsTr) if f.endswith(".nii.gz")]
series_all = sorted(f.replace(".nii.gz", "") for f in label_files)

positive_series = []
negative_series = []

for uid in series_all:
    lab_path = os.path.join(labelsTr, uid + ".nii.gz")
    lab = nib.load(lab_path).get_fdata()
    if lab.max() > 0:
        positive_series.append(uid)
    else:
        negative_series.append(uid)

print("Total series:", len(series_all))
print("Positive series:", len(positive_series))
print("Negative series:", len(negative_series))
print("Example positives:", positive_series[:5])
print("Example negatives:", negative_series[:5])


Total series: 100
Positive series: 45
Negative series: 55
Example positives: ['1.2.826.0.1.3680043.8.498.10743199796364362163988736837321335182', '1.2.826.0.1.3680043.8.498.10925835367566060680558681418372812622', '1.2.826.0.1.3680043.8.498.10935907012185032169927418164924236382', '1.2.826.0.1.3680043.8.498.11362594742849845937638082003271998271', '1.2.826.0.1.3680043.8.498.11803650639437261093226018672112512148']
Example negatives: ['1.2.826.0.1.3680043.8.498.10163482612339017493097015030860956863', '1.2.826.0.1.3680043.8.498.10454754803302367695534484904787098586', '1.2.826.0.1.3680043.8.498.10488876862972997983660376855639751518', '1.2.826.0.1.3680043.8.498.10489902145908525186969095759982595916', '1.2.826.0.1.3680043.8.498.10777851323461603684026638438811329191']


In [22]:
# pick a smaller subset, e.g. 40 cases

import numpy as np

rng = np.random.default_rng(42)  # reproducible

max_total   = 40
max_pos_keep = min(30, len(positive_series))
max_neg_keep = min(max_total - max_pos_keep, len(negative_series))

keep_pos = list(rng.choice(positive_series, size=max_pos_keep, replace=False)) if max_pos_keep > 0 else []
keep_neg = list(rng.choice(negative_series, size=max_neg_keep, replace=False)) if max_neg_keep > 0 else []

keep_series = sorted(set(keep_pos) | set(keep_neg))

print("Will keep total:", len(keep_series))
print("  positives:", len(keep_pos))
print("  negatives:", len(keep_neg))
print("First few kept:", keep_series[:10])

Will keep total: 40
  positives: 30
  negatives: 10
First few kept: ['1.2.826.0.1.3680043.8.498.10777851323461603684026638438811329191', '1.2.826.0.1.3680043.8.498.10925835367566060680558681418372812622', '1.2.826.0.1.3680043.8.498.10935907012185032169927418164924236382', '1.2.826.0.1.3680043.8.498.11803650639437261093226018672112512148', '1.2.826.0.1.3680043.8.498.12140455131248066497632485642004879217', '1.2.826.0.1.3680043.8.498.13185867632993439286148270878470785779', '1.2.826.0.1.3680043.8.498.13363897289235267814753067983525010231', '1.2.826.0.1.3680043.8.498.14235780456654341433067740280108862123', '1.2.826.0.1.3680043.8.498.26156563278593519244496678124557921928', '1.2.826.0.1.3680043.8.498.29519031269697701842810294832452877113']


In [23]:
# delete images + labels for series I am not keeping

to_keep = set(keep_series)

# Delete extra labels
for uid in series_all:
    if uid not in to_keep:
        lab_path = os.path.join(labelsTr, uid + ".nii.gz")
        if os.path.exists(lab_path):
            os.remove(lab_path)
            print("[DEL LABEL]", lab_path)

# Delete extra images
image_files = [f for f in os.listdir(imagesTr) if f.endswith("_0000.nii.gz")]
for f in image_files:
    uid = f.replace("_0000.nii.gz", "")
    if uid not in to_keep:
        img_path = os.path.join(imagesTr, f)
        if os.path.exists(img_path):
            os.remove(img_path)
            print("[DEL IMAGE]", img_path)

# Sanity check
imagesTr_files = [f for f in os.listdir(imagesTr) if f.endswith("_0000.nii.gz")]
labelsTr_files = [f for f in os.listdir(labelsTr) if f.endswith(".nii.gz")]
print("After pruning - imagesTr:", len(imagesTr_files), "labelsTr:", len(labelsTr_files))

[DEL LABEL] /kaggle/working/nnUNet_raw/Dataset201_AneurysmCTA/labelsTr/1.2.826.0.1.3680043.8.498.10163482612339017493097015030860956863.nii.gz
[DEL LABEL] /kaggle/working/nnUNet_raw/Dataset201_AneurysmCTA/labelsTr/1.2.826.0.1.3680043.8.498.10454754803302367695534484904787098586.nii.gz
[DEL LABEL] /kaggle/working/nnUNet_raw/Dataset201_AneurysmCTA/labelsTr/1.2.826.0.1.3680043.8.498.10488876862972997983660376855639751518.nii.gz
[DEL LABEL] /kaggle/working/nnUNet_raw/Dataset201_AneurysmCTA/labelsTr/1.2.826.0.1.3680043.8.498.10489902145908525186969095759982595916.nii.gz
[DEL LABEL] /kaggle/working/nnUNet_raw/Dataset201_AneurysmCTA/labelsTr/1.2.826.0.1.3680043.8.498.10743199796364362163988736837321335182.nii.gz
[DEL LABEL] /kaggle/working/nnUNet_raw/Dataset201_AneurysmCTA/labelsTr/1.2.826.0.1.3680043.8.498.10994836313290465695172433969490116921.nii.gz
[DEL LABEL] /kaggle/working/nnUNet_raw/Dataset201_AneurysmCTA/labelsTr/1.2.826.0.1.3680043.8.498.11024186785729776851960279299394139142.nii.gz

In [24]:
# update dataset.json (numTraining)

import json

ds_json_path = os.path.join(dataset_dir, "dataset.json")

with open(ds_json_path) as f:
    ds = json.load(f)

ds["numTraining"] = len(keep_series)

with open(ds_json_path, "w") as f:
    json.dump(ds, f, indent=4)

print("Updated numTraining to", ds["numTraining"])

Updated numTraining to 40


## Run nnU-Net planning & preprocessing

In [25]:
# delete all old preprocessed output using 
# (e.g. -np 2 (2 cells below) results)

import shutil
import os

prep_dir = "/kaggle/working/nnUNet_preprocessed/Dataset201_AneurysmCTA"

if os.path.isdir(prep_dir):
    shutil.rmtree(prep_dir)
    print("Deleted old preprocessed folder:", prep_dir)
else:
    print("No old preprocessed folder found.")

Deleted old preprocessed folder: /kaggle/working/nnUNet_preprocessed/Dataset201_AneurysmCTA


In [26]:
# re-running preprocessing with 1 process
!export nnUNet_raw=/kaggle/working/nnUNet_raw && \
 export nnUNet_preprocessed=/kaggle/working/nnUNet_preprocessed && \
 export nnUNet_results=/kaggle/working/nnUNet_results && \
 nnUNetv2_plan_and_preprocess -d 201 --verify_dataset_integrity -c 3d_fullres -np 1

Fingerprint extraction...
Dataset201_AneurysmCTA
Using <class 'nnunetv2.imageio.simpleitk_reader_writer.SimpleITKIO'> as reader/writer

####################
verify_dataset_integrity Done. 
If you didn't see any error messages then your dataset is most likely OK!
####################

Using <class 'nnunetv2.imageio.simpleitk_reader_writer.SimpleITKIO'> as reader/writer
100%|███████████████████████████████████████████| 40/40 [01:25<00:00,  2.13s/it]
Experiment planning...

############################
INFO: You are using the old nnU-Net default planner. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Attempting to find 3d_lowres config. 
Current spacing: [0.64375    0.4526366  0.45565397]. 
Current patch size: (96, 160, 160). 
Current median shape: [330.09708738 497.08737864 493.68932039]
Attempting to find 3d_lowres config. 
Current spaci

In [27]:
# check preprocessed folder size
!du -sh /kaggle/working/nnUNet_preprocessed/Dataset201_AneurysmCTA

9.2G	/kaggle/working/nnUNet_preprocessed/Dataset201_AneurysmCTA


In [17]:
# # preprocessing with 2 processes
# !export nnUNet_raw=/kaggle/working/nnUNet_raw && \
#  export nnUNet_preprocessed=/kaggle/working/nnUNet_preprocessed && \
#  export nnUNet_results=/kaggle/working/nnUNet_results && \
#  nnUNetv2_plan_and_preprocess -d 201 --verify_dataset_integrity -c 3d_fullres -np 2

Fingerprint extraction...
Dataset201_AneurysmCTA
Using <class 'nnunetv2.imageio.simpleitk_reader_writer.SimpleITKIO'> as reader/writer

####################
verify_dataset_integrity Done. 
If you didn't see any error messages then your dataset is most likely OK!
####################

Using <class 'nnunetv2.imageio.simpleitk_reader_writer.SimpleITKIO'> as reader/writer
100%|█████████████████████████████████████████| 100/100 [03:51<00:00,  2.31s/it]
Experiment planning...

############################
INFO: You are using the old nnU-Net default planner. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Attempting to find 3d_lowres config. 
Current spacing: [0.64375    0.48242625 0.48283262]. 
Current patch size: (128, 128, 128). 
Current median shape: [433.98058252 497.08737864 496.60194175]
Attempting to find 3d_lowres config. 
Current spac